# RPG Eval-Seed Result Tables

This lightweight notebook loads eval-seed outputs from `artifacts/rpg/eval_seeds` and builds reporting tables for all available datasets and config tracks.

The eval script stays generic: it stores only per-user/per-seed metric rows and generic summaries. Paper targets and margin-specific checks such as relative ±5% mean matching and TOST equivalence are computed here, so changing the reporting criterion does not require rerunning Snellius jobs.

Expected result layout:

```text
artifacts/rpg/eval_seeds/<track>/<dataset>/<session>/summary.json
artifacts/rpg/eval_seeds/<track>/<dataset>/<session>/per_user_metrics.csv
```


In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

ROOT = Path.cwd()
ARTIFACT_ROOT = ROOT / "artifacts" / "rpg" / "eval_seeds"

REPORT_METRIC = "ndcg@10"
RELATIVE_EQ_MARGIN = 0.05  # 5% of the paper value.
TOST_CI_LEVEL = 0.90      # Standard two one-sided tests equivalence CI.
BOOTSTRAP_SAMPLES = 5000
BOOTSTRAP_SEED = 2024

PAPER_VALUES = {
    "ndcg@10": {
        "Sports": 0.0263,
        "Beauty": 0.0464,
        "Toys": 0.0490,
        "CDs": 0.0415,
    },
}

# If you run this notebook from another working directory, uncomment and set manually.
# ARTIFACT_ROOT = Path("/gpfs/home6/$USER/RPG/artifacts/rpg/eval_seeds").expanduser()

ARTIFACT_ROOT


In [ ]:
TRACK_LABELS = {
    "released_readme": "released README",
    "paper_appendix": "paper appendix",
}

DATASET_LABELS = {
    "Sports_and_Outdoors": "Sports",
    "sports_and_outdoors": "Sports",
    "Beauty": "Beauty",
    "beauty": "Beauty",
    "Toys_and_Games": "Toys",
    "toys_and_games": "Toys",
    "CDs_and_Vinyl": "CDs",
    "cds_and_vinyl": "CDs",
}


def _metric_summary(payload: dict, metric: str) -> dict:
    for row in payload.get("metric_summary", []):
        if row.get("metric") == metric:
            return row
    raise KeyError(f"Metric {metric!r} not found in metric_summary")


def _paper_value(dataset: str, metric: str) -> float | None:
    value = PAPER_VALUES.get(metric, {}).get(dataset)
    return None if value is None else float(value)


def _mean_visited_items(payload: dict) -> float:
    values = [row.get("n_visited_items") for row in payload.get("per_seed_summary", [])]
    values = [float(value) for value in values if value is not None]
    return float(np.mean(values)) if values else math.nan


def _track_and_dataset(summary_path: Path) -> tuple[str, str]:
    rel_parts = summary_path.relative_to(ARTIFACT_ROOT).parts
    if len(rel_parts) >= 4 and rel_parts[0] in TRACK_LABELS:
        return rel_parts[0], rel_parts[1]
    if len(rel_parts) >= 3:
        return "unknown", rel_parts[0]
    return "unknown", "unknown"


def _bootstrap_ci(values: np.ndarray, *, n_samples: int, seed: int, ci_level: float, offset: float = 0.0) -> tuple[float, float]:
    values = np.asarray(values, dtype=np.float64)
    values = values[~np.isnan(values)]
    if values.size == 0:
        return math.nan, math.nan
    if n_samples <= 0:
        estimate = float(values.mean() - offset)
        return estimate, estimate

    rng = np.random.default_rng(seed)
    estimates = np.empty(n_samples, dtype=np.float64)
    for index in range(n_samples):
        sample_indices = rng.integers(0, values.shape[0], size=values.shape[0])
        estimates[index] = values[sample_indices].mean() - offset

    alpha = 1.0 - ci_level
    low, high = np.quantile(estimates, [alpha / 2.0, 1.0 - alpha / 2.0])
    return float(low), float(high)


def _per_user_seed_means(summary_path: Path, metric: str) -> np.ndarray | None:
    per_user_path = summary_path.parent / "per_user_metrics.csv"
    if not per_user_path.exists():
        return None
    per_user = pd.read_csv(per_user_path, usecols=["user_index", "eval_seed", metric])
    per_user[metric] = per_user[metric].astype(float)
    z_user = per_user.groupby("user_index", sort=True)[metric].mean()
    return z_user.to_numpy(dtype=np.float64)


def _relative_equivalence(summary_path: Path, metric: str, ours: float, paper: float | None) -> dict:
    if paper is None or paper == 0:
        return {
            "relative_margin_abs": math.nan,
            "relative_diff_pct": math.nan,
            "mean_within_relative_margin": None,
            "tost_diff_ci_low": math.nan,
            "tost_diff_ci_high": math.nan,
            "tost_equivalent_relative_margin": None,
            "tost_source": "missing paper value",
        }

    margin = abs(paper) * RELATIVE_EQ_MARGIN
    diff = ours - paper
    output = {
        "relative_margin_abs": margin,
        "relative_diff_pct": 100.0 * diff / paper,
        "mean_within_relative_margin": abs(diff) <= margin,
        "tost_diff_ci_low": math.nan,
        "tost_diff_ci_high": math.nan,
        "tost_equivalent_relative_margin": None,
        "tost_source": "missing per_user_metrics.csv",
    }

    z_user = _per_user_seed_means(summary_path, metric)
    if z_user is None:
        return output

    diff_low, diff_high = _bootstrap_ci(
        z_user,
        n_samples=BOOTSTRAP_SAMPLES,
        seed=BOOTSTRAP_SEED,
        ci_level=TOST_CI_LEVEL,
        offset=paper,
    )
    output.update(
        {
            "tost_diff_ci_low": diff_low,
            "tost_diff_ci_high": diff_high,
            "tost_equivalent_relative_margin": diff_low >= -margin and diff_high <= margin,
            "tost_source": "per_user_metrics.csv",
        }
    )
    return output


def load_eval_seed_runs(metric: str = REPORT_METRIC) -> pd.DataFrame:
    if not ARTIFACT_ROOT.exists():
        return pd.DataFrame()

    rows = []
    for summary_path in sorted(ARTIFACT_ROOT.rglob("summary.json")):
        payload = json.loads(summary_path.read_text())
        track, dataset_slug = _track_and_dataset(summary_path)
        dataset = DATASET_LABELS.get(payload.get("category"), DATASET_LABELS.get(dataset_slug, dataset_slug))
        metric_row = _metric_summary(payload, metric=metric)
        paper = _paper_value(dataset, metric)
        ours = float(metric_row["final_user_avg"])
        diff = None if paper is None else ours - paper
        eval_seed_std = float(metric_row.get("eval_seed_std", math.nan))
        diff_over_seed_std = math.nan
        if diff is not None and eval_seed_std > 0:
            diff_over_seed_std = abs(diff) / eval_seed_std

        ci_low = float(metric_row.get("user_bootstrap_ci_low", math.nan))
        ci_high = float(metric_row.get("user_bootstrap_ci_high", math.nan))
        paper_inside_ci = None
        if paper is not None and not math.isnan(ci_low) and not math.isnan(ci_high):
            paper_inside_ci = ci_low <= paper <= ci_high

        rel_eq = _relative_equivalence(summary_path, metric, ours, paper)

        rows.append(
            {
                "track": track,
                "config_source": TRACK_LABELS.get(track, track),
                "dataset_slug": dataset_slug,
                "dataset": dataset,
                "session": summary_path.parent.name,
                "summary_path": str(summary_path),
                "checkpoint_path": payload.get("checkpoint_path"),
                "n_eval_seeds": len(payload.get("eval_seeds", [])),
                "n_users": int(metric_row.get("n_users", 0)),
                "metric": metric,
                "paper_ndcg10": paper,
                "ours_ndcg10": ours,
                "diff": diff,
                "eval_seed_std": eval_seed_std,
                "diff_over_seed_std": diff_over_seed_std,
                "user_ci_level": float(metric_row.get("user_bootstrap_ci_level", math.nan)),
                "user_ci_low": ci_low,
                "user_ci_high": ci_high,
                "paper_inside_user_ci": paper_inside_ci,
                "mean_visited_items": _mean_visited_items(payload),
                "bootstrap_samples_summary": int(payload.get("bootstrap_samples", 0)),
                "bootstrap_seed_summary": int(payload.get("bootstrap_seed", 0)),
                **rel_eq,
            }
        )

    return pd.DataFrame(rows)


all_runs = load_eval_seed_runs(REPORT_METRIC)
all_runs


In [ ]:
if all_runs.empty:
    print(f"No summary.json files found under {ARTIFACT_ROOT}")
else:
    # Keep the newest-looking session for each dataset/config source.
    latest = (
        all_runs.sort_values(["dataset", "track", "session"])
        .groupby(["dataset", "track"], as_index=False)
        .tail(1)
        .sort_values(["dataset", "track"])
        .reset_index(drop=True)
    )
    display(latest[["dataset", "config_source", "session", "summary_path"]])


## Table 1: Reproduction Accuracy

This table focuses on fixed-split reproduction: how far our `NDCG@10` is from the paper, whether the mean is inside the pre-specified relative tolerance, and whether a stricter user-bootstrap TOST establishes equivalence within the same relative tolerance.


In [ ]:
if not all_runs.empty:
    table_reproduction = latest[
        [
            "dataset",
            "config_source",
            "paper_ndcg10",
            "ours_ndcg10",
            "diff",
            "relative_diff_pct",
            "relative_margin_abs",
            "mean_within_relative_margin",
            "tost_equivalent_relative_margin",
            "eval_seed_std",
            "diff_over_seed_std",
        ]
    ].copy()
    table_reproduction = table_reproduction.rename(
        columns={
            "dataset": "Dataset",
            "config_source": "Config source",
            "paper_ndcg10": "Paper NDCG@10",
            "ours_ndcg10": "Ours NDCG@10",
            "diff": "Diff",
            "relative_diff_pct": "Rel. diff %",
            "relative_margin_abs": f"±{RELATIVE_EQ_MARGIN:.0%} margin",
            "mean_within_relative_margin": f"Mean within ±{RELATIVE_EQ_MARGIN:.0%}?",
            "tost_equivalent_relative_margin": f"TOST equivalent ±{RELATIVE_EQ_MARGIN:.0%}?",
            "eval_seed_std": "Eval seed std",
            "diff_over_seed_std": "|Diff| / seed std",
        }
    )
    display(table_reproduction.style.format({
        "Paper NDCG@10": "{:.4f}",
        "Ours NDCG@10": "{:.5f}",
        "Diff": "{:+.5f}",
        "Rel. diff %": "{:+.1f}%",
        f"±{RELATIVE_EQ_MARGIN:.0%} margin": "{:.5f}",
        "Eval seed std": "{:.6f}",
        "|Diff| / seed std": "{:.1f}",
    }))


## Table 2: TOST Details

The TOST column above uses a bootstrap CI of the difference over users after averaging each user over eval seeds. Equivalence passes only if the whole CI is inside `[-margin, +margin]`.


In [ ]:
if not all_runs.empty:
    table_tost = latest[
        [
            "dataset",
            "config_source",
            "diff",
            "relative_margin_abs",
            "tost_diff_ci_low",
            "tost_diff_ci_high",
            "mean_within_relative_margin",
            "tost_equivalent_relative_margin",
            "tost_source",
        ]
    ].copy()
    table_tost[f"{TOST_CI_LEVEL:.0%} TOST diff CI"] = table_tost.apply(
        lambda row: "missing" if pd.isna(row["tost_diff_ci_low"]) else f"[{row['tost_diff_ci_low']:+.5f}, {row['tost_diff_ci_high']:+.5f}]",
        axis=1,
    )
    table_tost = table_tost[
        [
            "dataset",
            "config_source",
            "diff",
            "relative_margin_abs",
            f"{TOST_CI_LEVEL:.0%} TOST diff CI",
            "mean_within_relative_margin",
            "tost_equivalent_relative_margin",
            "tost_source",
        ]
    ].rename(
        columns={
            "dataset": "Dataset",
            "config_source": "Config source",
            "diff": "Diff",
            "relative_margin_abs": f"±{RELATIVE_EQ_MARGIN:.0%} margin",
            "mean_within_relative_margin": f"Mean within ±{RELATIVE_EQ_MARGIN:.0%}?",
            "tost_equivalent_relative_margin": f"TOST equivalent ±{RELATIVE_EQ_MARGIN:.0%}?",
            "tost_source": "TOST source",
        }
    )
    display(table_tost.style.format({
        "Diff": "{:+.5f}",
        f"±{RELATIVE_EQ_MARGIN:.0%} margin": "{:.5f}",
    }))


## Table 3: User-Level Uncertainty

This table reports the classical IR-style uncertainty over users. It should be interpreted separately from eval-seed variance and the equivalence margin.


In [ ]:
if not all_runs.empty:
    table_user_ci = latest[
        [
            "dataset",
            "config_source",
            "ours_ndcg10",
            "user_ci_low",
            "user_ci_high",
            "paper_inside_user_ci",
            "bootstrap_samples_summary",
            "bootstrap_seed_summary",
        ]
    ].copy()
    table_user_ci["User bootstrap CI"] = table_user_ci.apply(
        lambda row: f"[{row['user_ci_low']:.5f}, {row['user_ci_high']:.5f}]",
        axis=1,
    )
    table_user_ci = table_user_ci[
        ["dataset", "config_source", "ours_ndcg10", "User bootstrap CI", "paper_inside_user_ci", "bootstrap_samples_summary", "bootstrap_seed_summary"]
    ].rename(
        columns={
            "dataset": "Dataset",
            "config_source": "Config source",
            "ours_ndcg10": "Ours NDCG@10",
            "paper_inside_user_ci": "Paper inside user CI?",
            "bootstrap_samples_summary": "Summary bootstrap samples",
            "bootstrap_seed_summary": "Summary bootstrap seed",
        }
    )
    display(table_user_ci.style.format({"Ours NDCG@10": "{:.5f}"}))


## Appendix Table: Full Run Metadata

Useful for checking which checkpoint/session produced each row.


In [ ]:
if not all_runs.empty:
    metadata_cols = [
        "dataset",
        "config_source",
        "session",
        "n_eval_seeds",
        "n_users",
        "mean_visited_items",
        "checkpoint_path",
        "summary_path",
    ]
    display(latest[metadata_cols].style.format({"mean_visited_items": "{:.1f}"}))
